# **LAB DL Session 4: Word Embedding Techniques**
Translasi kata ke dalam vektor numerik yang dapat digunakan oleh model machine learning.
untuk prediksi teks2 berikutnya

Step 1: Tokenization
- Data Raw -> Tokenization -> List of Tokens
- Standarisasi adalah proses mengubah teks menjadi bentuk yang lebih konsisten, misalnya mengubah semua huruf menjadi huruf kecil, menghapus tanda baca, dan menghapus karakter khusus.
- Contoh: "Saya suka makan nasi goreng." -> "saya", "suka", "makan", "nasi", "goreng"
- Padding 0 = kasih tau gada isi

Step 2: Stop Words Removal
- Stop Words adalah kata-kata umum yang sering muncul dalam teks tetapi tidak memberikan banyak informasi, seperti "dan", "atau", "adalah", dll. Proses ini melibatkan menghapus kata-kata tersebut dari daftar token.
- List of Tokens -> Stop Words Removal -> Cleaned Tokens

RNN : Read Word by Word
RNN juga punya masalah, ex di ai agent suka lupa context. 

RNN maksa untuk baca dataset yang besar, tapi RNN punya masalah dengan long-term dependencies. Jadi kalau ada kata yang penting di awal, RNN bisa lupa karena terlalu banyak informasi yang harus diingat.

Maka, pake LSTM (Long Short-Term Memory) atau GRU (Gated Recurrent Unit) untuk mengatasi masalah ini. LSTM dan GRU memiliki mekanisme untuk mengingat informasi penting dalam jangka panjang, sehingga lebih efektif dalam menangani konteks yang panjang.


In [ ]:
# # Uncomment jika pakai google colab karena Google Colab tidak memiliki semua library yang dibutuhkan.
# from google.colab import files
# import zipfile
# import os

# # 1. Upload file zip dari laptop
# print("Pilih file ZIP dataset:")
# uploaded = files.upload()

# print("Selesai! Struktur dataset:")

In [13]:
# Import library

import tensorflow as tf
import re
import numpy as np
from sklearn.model_selection import train_test_split

In [14]:
# Preprocessing Sentence

# Load dataset sentence sebelum diconvert jadi token

def preprocess_sentence(sentence):
  # 1. Lowercase +strip
  sentence = sentence.lower().strip()
  # 2. Tampahin spasi setelah punctuation
  # sub(pattern, replacement, text)
  sentence = re.sub(r"([?.,!])", r"\1 ", sentence)
  # 3. Hapus spasi yang berlebih
  sentence = re.sub('[" "]+', " ", sentence)
  # 4. Hpaus karakter selain A-Z dan tanda baca
  sentence = re.sub(r"[^a-zA-Z?.,!]", " ", sentence)
  # 5. Hapus spasi diawal dan di akhir
  sentence = sentence.strip()
  # 6. Kasih tanda awal dan akhir dari sequence
  sentence = '<start> ' + sentence + ' <end>'

  return sentence


In [15]:
# Load Data
def load_dataset(path, num_examples=None):
  with open(path, 'r', encoding='utf-8') as file:
    lines = file.read().strip().split('\n')

  parts = [line.split('\t') for line in lines[:num_examples]]

  input_data = [preprocess_sentence(part[0]) for part in parts]
  target_data = [preprocess_sentence(part[1]) for part in parts]

  return input_data, target_data

path_to_file = '/content/sample_data/ind.txt'
input_data, target_data = load_dataset(path_to_file, num_examples=100000)

print("Input Data : ", input_data)
print("Target Data : ", target_data)

Input Data :  ['<start> hi. <end>', '<start> run! <end>', '<start> run. <end>', '<start> who? <end>', '<start> wow! <end>', '<start> wow! <end>', '<start> fire! <end>', '<start> fire! <end>', '<start> fire! <end>', '<start> help! <end>', '<start> jump! <end>', '<start> jump. <end>', '<start> stop! <end>', '<start> stop! <end>', '<start> wait! <end>', '<start> wait. <end>', '<start> hello. <end>', '<start> hurry! <end>', '<start> hurry! <end>', '<start> hurry! <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i won! <end>', '<start> i won. <end>', '<start> oh no! <end>', '<start> relax. <end>', '<start> relax. <end>', '<start> shoot! <end>', '<start> smile. <end>', '<start> attack! <end>', '<start> cheers! <end>', '<start> freeze! <end>', '<start> get up. <end>', '<start> got it! <end>', '<start> got it? <end>', '<start> he ran. <end>', '<start> i fell. <end>', '<start> i know.

In [16]:
# Tokenize input and target data

# 1. Create tokenizer object
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')

# 2. Tokenize the data
tokenizer.fit_on_texts(input_data + target_data)

In [17]:
# Convert Sentences into Integer Sequences

input_tensor = tokenizer.texts_to_sequences(input_data)
target_tensor = tokenizer.texts_to_sequences(target_data)

In [18]:
# Calculate Max Sequence Length

max_length_targ, max_length_inp = max(len(seq) for seq in target_tensor), max(len(seq) for seq in input_tensor)

In [19]:
# Padding Sequence

input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_tensor, maxlen=max_length_inp, padding='post') #post -> akhir dari sequence

target_tensor = tf.keras.preprocessing.sequence.pad_sequences(target_tensor, maxlen=max_length_targ, padding='post')

In [20]:
# Calculate Vocabulary Size

vocab_size = len(tokenizer.word_index) + 1

print("Input Tensor : ", input_tensor)
print("Target Tensor : ", target_tensor)

Input Tensor :  [[    1  4450     2 ...     0     0     0]
 [    1  8571     2 ...     0     0     0]
 [    1  5763     2 ...     0     0     0]
 ...
 [    1 12056 12057 ...     0     0     0]
 [    1  7194  7195 ...     0     0     0]
 [    1   152    11 ...    27  2522     2]]
Target Tensor :  [[    1  5085     2 ...     0     0     0]
 [    1  7197     2 ...     0     0     0]
 [    1  7197     2 ...     0     0     0]
 ...
 [    1 16335 16336 ...     0     0     0]
 [    1  7194  7195 ...     0     0     0]
 [    1   277   614 ...     0     0     0]]


In [21]:
# Splitting data
# 80 Train, 20 Test

input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.2, random_state=42)

BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)

dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

print('Dataset : ', dataset)

for sample in dataset.take(1):
    input_sample, target_sample = sample
    print('Input Sample : ', input_sample)
    print('Target Sample : ', target_sample)

Dataset :  <_BatchDataset element_spec=(TensorSpec(shape=(64, 36), dtype=tf.int32, name=None), TensorSpec(shape=(64, 30), dtype=tf.int32, name=None))>
Input Sample :  tf.Tensor(
[[   1    4   93 ...    0    0    0]
 [   1    3  120 ...    0    0    0]
 [   1    4   38 ...    0    0    0]
 ...
 [   1    3   99 ...    0    0    0]
 [   1    4   38 ...    0    0    0]
 [   1    3 4521 ...    0    0    0]], shape=(64, 36), dtype=int32)
Target Sample :  tf.Tensor(
[[    1     6    59 ...     0     0     0]
 [    1  1097   158 ...     0     0     0]
 [    1     6 14344 ...     0     0     0]
 ...
 [    1     3    10 ...     0     0     0]
 [    1     6  2362 ...     0     0     0]
 [    1     3  4098 ...     0     0     0]], shape=(64, 30), dtype=int32)


In [22]:
# Class Encoder

class Encoder(tf.keras.Model):

  # Constructor
  def __init__(self, vocab_size, embedding_dim, enc_units, batch_size):
    super(Encoder, self).__init__()
    self.batch_size = batch_size
    self.enc_units = enc_units
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(self.enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

  # Call -> pakai model pada data yang diberikan
  def call(self, x, hidden):
    x = self.embedding(x)
    output, state = self.gru(x, initial_state=hidden)
    return output, state

  # Initilize hidden state
  def init_hidden_state(self):
    return tf.zeros((self.batch_size, self.enc_units))


In [23]:
# Initialize value2 yg digunakan untuk encoder

embedding_dim = 256

units = 1024
BATCH_SIZE = 64
example_input_batch = input_tensor_train[:BATCH_SIZE]

# create instance encoder
encoder = Encoder(vocab_size, embedding_dim, units, BATCH_SIZE)

sample_hidden = encoder.init_hidden_state()
sample_output, sample_hidden = encoder(example_input_batch, sample_hidden)

print('Encoder Output Shape (batch size, seq length, units) : {}'.format(sample_output.shape))
print('Encoder Hidden Shape (batch size, units) : {}'.format(sample_hidden.shape))

Encoder Output Shape (batch size, seq length, units) : (64, 36, 1024)
Encoder Hidden Shape (batch size, units) : (64, 1024)
